In [ ]:
"""
plot_single.py — Generate individual thesis-quality figures from a single pump video.

Each panel is saved as a separate PDF and PNG.
Reuses load_signals / compute_features from pump_extractor.py.

Usage
-----
    python plot_single.py

Edit the CONFIG block below.
"""

from __future__ import annotations

import json
import logging
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from scipy.signal import find_peaks, hilbert, welch
from scipy.fft import rfft, rfftfreq

# ── paste your pump_extractor constants here ──────────────────────────────────
FPS_DEFAULT          = 30.0
DIFF_THRESH_FACTOR   = 0.25
DIFF_THRESH_MIN      = 20
PUMP_PERSIST_FRAMES  = 6
PEAK_MIN_DIST_SEC    = 0.07
PEAK_PROM_FACTOR     = 0.35
FREQ_WIN_SEC         = 4.0
FREQ_STEP_SEC        = 0.5
FREQ_LO_HZ           = 1.0
FREQ_HI_HZ           = 20.0
FLOW_DOWNSAMPLE      = 0.5
ACTIVITY_MASK_FACTOR = 0.5
CLAHE_CLIP           = 2.0
CLAHE_GRID           = (4, 4)
SHARPNESS_MIN        = 20.0

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG  ← edit these
# ══════════════════════════════════════════════════════════════════════════════

VIDEO_PATH = "data/<path-to-your-video>.mp4"   # single video -- point at any pump recording
ROI_JSON   = "data/training/normal_pump_rois.json"     # same JSON used in pump_extractor
OUT_DIR    = "other_figures/output_figures"   # folder; will be created if missing

# matplotlib style
FONT_FAMILY = "serif"          # matches thesis font; change to "sans-serif" if needed
FONT_SIZE   = 11
LABEL_SIZE  = 10
DPI_PNG     = 200
C           = "#2E7D32"        # main plot colour
C_ANOM      = "#C62828"        # secondary / anomaly colour

# ══════════════════════════════════════════════════════════════════════════════

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger(__name__)

# ── copy load_signals and compute_features verbatim from pump_extractor.py ───
# (included here so the script is self-contained)

def load_signals(video_path: str, roi: tuple) -> dict | None:
    x1, y1, x2, y2 = roi
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        log.error("Cannot open: %s", video_path)
        return None
    fps = float(cap.get(cv2.CAP_PROP_FPS)) or FPS_DEFAULT
    roi_frames, raw_roi_samples = [], []
    i_frame = 0
    ret, frame = cap.read()
    while ret:
        gray_raw = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        g = cv2.GaussianBlur(gray_raw, (5, 5), 0)
        roi_frames.append(g[y1:y2, x1:x2].copy())
        if i_frame % 300 == 0:
            raw_roi_samples.append(gray_raw[y1:y2, x1:x2].copy())
        i_frame += 1
        ret, frame = cap.read()
    cap.release()
    N = len(roi_frames)
    if N < 60:
        return None
    H, W = roi_frames[0].shape
    sharpness = float(np.mean([cv2.Laplacian(r, cv2.CV_64F).var() for r in raw_roi_samples])) if raw_roi_samples else 999.0
    if sharpness < SHARPNESS_MIN:
        log.warning("REJECTED blurry (sharpness=%.1f)", sharpness)
        return None
    roi_sample = np.stack(roi_frames[N//4: N//4+30]).mean(axis=0)
    diff_thresh = max(DIFF_THRESH_MIN, int(roi_sample.mean() * DIFF_THRESH_FACTOR))
    roi_mot = np.zeros(N, dtype=np.float32)
    for i in range(1, N):
        d = cv2.absdiff(roi_frames[i], roi_frames[i-1])
        _, bm = cv2.threshold(d, 15, 255, cv2.THRESH_BINARY)
        roi_mot[i] = float(np.count_nonzero(cv2.medianBlur(bm, 5)))
    roi_mot_thresh = max(10, int(H * W * 0.05))
    ps = pe = None
    streak = 0
    for i, v in enumerate(roi_mot):
        if v >= roi_mot_thresh:
            streak += 1
            if streak >= PUMP_PERSIST_FRAMES and ps is None:
                ps = i - PUMP_PERSIST_FRAMES + 1
        else:
            streak = 0
    pe = next((i for i in range(N-1, -1, -1) if roi_mot[i] >= roi_mot_thresh), None)
    if ps is None or pe is None or pe <= ps:
        return None
    roi_seg = roi_frames[ps: pe+1]
    T = len(roi_seg)
    frames_f = np.stack([f.astype(np.float32) for f in roi_seg])
    var_map = frames_f.var(axis=0)
    active_mask = var_map > var_map.mean() * ACTIVITY_MASK_FACTOR
    active_count = max(1, int(active_mask.sum()))
    diff_sig = np.zeros(T, dtype=np.float32)
    top_d = np.zeros(T, dtype=np.float32)
    bot_d = np.zeros(T, dtype=np.float32)
    left_d = np.zeros(T, dtype=np.float32)
    right_d = np.zeros(T, dtype=np.float32)
    for i in range(1, T):
        d = cv2.absdiff(roi_seg[i], roi_seg[i-1])
        _, bm = cv2.threshold(d, diff_thresh, 255, cv2.THRESH_BINARY)
        bm_med = cv2.medianBlur(bm, 5)
        diff_sig[i] = float(np.count_nonzero(bm_med[active_mask])) / active_count
        top_d[i] = float(np.mean(d[:H//2, :]))
        bot_d[i] = float(np.mean(d[H//2:, :]))
        left_d[i] = float(np.mean(d[:, :W//2]))
        right_d[i] = float(np.mean(d[:, W//2:]))
    flow_sig = np.zeros(T, dtype=np.float32)
    ds = FLOW_DOWNSAMPLE
    prev = cv2.resize(roi_seg[0], (max(8, int(W*ds)), max(8, int(H*ds))))
    diag = float(np.hypot(prev.shape[0], prev.shape[1]) + 1e-6)
    fb = dict(pyr_scale=0.5, levels=3, winsize=20, iterations=3, poly_n=5, poly_sigma=1.7, flags=0)
    for i in range(1, T):
        cur = cv2.resize(roi_seg[i], prev.shape[::-1])
        fl = cv2.calcOpticalFlowFarneback(prev, cur, None, **fb)
        mag, _ = cv2.cartToPolar(fl[..., 0], fl[..., 1])
        flow_sig[i] = float(np.mean(np.clip(mag, 0, np.percentile(mag, 99))) / diag)
        prev = cur
    return dict(fps=fps, T=T, H=H, W=W,
                pump_start_s=ps/fps, pump_end_s=pe/fps, pump_dur_s=(pe-ps)/fps,
                sharpness=sharpness, diff=diff_sig, flow=flow_sig,
                top=top_d, bot=bot_d, left=left_d, right=right_d,
                frames=frames_f, diff_thresh=diff_thresh,
                active_mask=active_mask, active_count=active_count)


def compute_features(sig: dict) -> dict:
    fps = sig["fps"]; seg = sig["diff"]; flow = sig["flow"]
    top = sig["top"]; bot = sig["bot"]; left = sig["left"]; right = sig["right"]
    frames = sig["frames"]; T, H, W = frames.shape
    f = {}
    activity_mask = sig.get("active_mask", frames.var(axis=0) > frames.var(axis=0).mean() * ACTIVITY_MASK_FACTOR)
    fw, psd = welch(seg, fps, nperseg=min(512, len(seg)//4))
    bw_mask = (fw >= FREQ_LO_HZ) & (fw <= FREQ_HI_HZ)
    if np.any(bw_mask):
        peak_idx = int(np.argmax(psd[bw_mask]))
        dom = float(fw[bw_mask][peak_idx])
        half_max = psd[bw_mask][peak_idx] / 2.0
        fw_band = fw[bw_mask]; psd_band = psd[bw_mask]
        left_i  = next((i for i in range(peak_idx, -1, -1) if psd_band[i] < half_max), 0)
        right_i = next((i for i in range(peak_idx, len(fw_band)) if psd_band[i] < half_max), len(fw_band)-1)
        bandwidth = max(fw_band[right_i] - fw_band[left_i], dom * 0.20)
        lo = dom - bandwidth; hi = dom + bandwidth
    else:
        dom = lo = hi = 0.0
    f["dominant_freq_hz"] = dom
    s_full = seg - seg.mean()
    yf = np.abs(rfft(s_full)); xf = rfftfreq(len(s_full), 1.0/fps)
    ab = (xf >= FREQ_LO_HZ) & (xf <= FREQ_HI_HZ)
    pb = (xf >= lo) & (xf <= hi)
    total_pw = float(np.sum(yf[ab]**2)) + 1e-9
    f["spectral_power_in_band"] = float(np.sum(yf[pb]**2)) / total_pw
    f["spectral_snr"] = float(yf[ab].max() / (np.median(yf[ab]) + 1e-9)) if np.any(ab) else 0.0
    psd_norm = psd[bw_mask] / (psd[bw_mask].sum() + 1e-9)
    f["spectral_entropy"] = float(-np.sum(psd_norm * np.log(psd_norm + 1e-12)))
    ph2 = (xf >= dom*1.8) & (xf <= dom*2.2)
    f["harmonic2_ratio"] = float(np.sum(yf[ph2]**2)) / (float(np.sum(yf[pb]**2)) + 1e-9)
    wf2 = int(fps * FREQ_WIN_SEC); sf2 = int(fps * FREQ_STEP_SEC)
    tv_f = []
    for start in range(0, len(seg)-wf2+1, sf2):
        c = seg[start:start+wf2] - seg[start:start+wf2].mean()
        yfc = np.abs(rfft(c)); xfc = rfftfreq(len(c), 1.0/fps)
        ba = (xfc >= FREQ_LO_HZ) & (xfc <= FREQ_HI_HZ)
        tv_f.append(float(xfc[ba][np.argmax(yfc[ba])]) if np.any(ba) else 0.0)
    tv_arr = np.array(tv_f)
    f["freq_stability_std"] = float(tv_arr.std())
    peaks, _ = find_peaks(seg, distance=max(1, int(PEAK_MIN_DIST_SEC*fps)), prominence=seg.std()*PEAK_PROM_FACTOR)
    valleys, _ = find_peaks(-seg, distance=max(1, int(PEAK_MIN_DIST_SEC*fps)))
    if len(peaks) > 1:
        ivls = np.diff(peaks) / fps * 1000
        f["interval_cv"] = float(ivls.std() / (ivls.mean() + 1e-9))
    else:
        ivls = np.array([0.0]); f["interval_cv"] = 0.0
    amps = []
    for pk in peaks:
        lv = valleys[valleys < pk]; rv = valleys[valleys > pk]
        if len(lv) and len(rv):
            amps.append(float(seg[pk] - min(seg[lv[-1]], seg[rv[0]])))
    amps_arr = np.array(amps) if amps else np.array([0.0])
    f["stroke_amp_cv"] = float(amps_arr.std() / (amps_arr.mean() + 1e-9))
    rise_t = []; fall_t = []
    for pk in peaks:
        lv = valleys[valleys < pk]; rv = valleys[valleys > pk]
        if len(lv): rise_t.append(float(pk - lv[-1]) / fps * 1000)
        if len(rv):  fall_t.append(float(rv[0] - pk) / fps * 1000)
    if rise_t and fall_t:
        f["rise_fall_ratio"] = float(np.mean(rise_t) / (np.mean(fall_t) + 1e-9))
        f["rise_time_cv"]    = float(np.std(rise_t) / (np.mean(rise_t) + 1e-9))
        f["fall_time_cv"]    = float(np.std(fall_t) / (np.mean(fall_t) + 1e-9))
    else:
        f["rise_fall_ratio"] = f["rise_time_cv"] = f["fall_time_cv"] = 0.0
    lag_p = int(round(fps / dom)) if dom > 0 else 3
    f["autocorr_at_period"] = float(np.corrcoef(seg[:-lag_p], seg[lag_p:])[0, 1]) if len(seg) > lag_p else 0.0
    if lag_p < len(seg):
        x_pp, y_pp = seg[:-lag_p], seg[lag_p:]
        eigs = np.linalg.eigvalsh(np.cov(x_pp, y_pp))
        f["phase_portrait_eccentricity"] = float(np.sqrt(1 - (eigs.min() / (eigs.max() + 1e-9))))
    else:
        f["phase_portrait_eccentricity"] = 0.0
    shapes = []
    if len(peaks) > 4 and dom > 0:
        for i in range(len(peaks)-1):
            cyc = seg[peaks[i]:peaks[i+1]]
            if len(cyc) > 3:
                shapes.append(np.interp(np.linspace(0, 1, lag_p), np.linspace(0, 1, len(cyc)), cyc))
    if len(shapes) > 3:
        ca = np.array(shapes)
        f["cycle_shape_cv"]  = float(ca.std(axis=0).mean() / (ca.mean(axis=0).mean() + 1e-9))
        cc = [float(np.corrcoef(ca[i], ca[i+1])[0, 1]) for i in range(len(ca)-1)]
        f["cycle_corr_mean"] = float(np.mean(cc))
    else:
        f["cycle_shape_cv"] = f["cycle_corr_mean"] = 0.0
    envelope = np.abs(hilbert(seg - seg.mean()))
    f["envelope_cv"] = float(envelope.std() / (envelope.mean() + 1e-9))
    f["top_bot_corr"] = float(np.corrcoef(top[1:], bot[1:])[0, 1])
    acl = activity_mask[:, :W//2].any(axis=0); acr = activity_mask[:, W//2:].any(axis=0)
    if acl.any() and acr.any():
        l_ts = frames[:, :, :W//2][:, :, acl].mean(axis=(1, 2))
        r_ts = frames[:, :, W//2:][:, :, acr].mean(axis=(1, 2))
        f["left_right_corr"] = float(np.corrcoef(l_ts, r_ts)[0, 1])
    else:
        f["left_right_corr"] = float(np.corrcoef(left[1:], right[1:])[0, 1])
    qcvs = []
    for ys, ye, xs, xe in [(0, H//2, 0, W//2), (0, H//2, W//2, W), (H//2, H, 0, W//2), (H//2, H, W//2, W)]:
        qd = np.array([float(np.mean(np.abs(frames[j, ys:ye, xs:xe] - frames[j-1, ys:ye, xs:xe]))) for j in range(1, T)])
        qcvs.append(float(qd.std() / (qd.mean() + 1e-9)))
    f["quadrant_cv_max"] = float(max(qcvs))
    f["active_roi_fraction"] = float(activity_mask.mean())
    f["motion_cv"] = float(seg.std() / (seg.mean() + 1e-9))
    fw_f, psd_f = welch(flow[1:], fps, nperseg=min(512, len(flow)//4))
    pump_band = (fw_f >= 7.0) & (fw_f <= 14.0)
    f["flow_dom_freq"] = float(fw_f[pump_band][np.argmax(psd_f[pump_band])]) if np.any(pump_band) else 0.0
    f["freq_flow_diff"] = abs(dom - f["flow_dom_freq"])
    # private helpers
    f["_peaks"] = peaks; f["_valleys"] = valleys; f["_tv_f"] = tv_arr
    f["_dom"] = dom; f["_lo"] = lo; f["_hi"] = hi
    f["_amps"] = amps_arr; f["_ivls"] = ivls; f["_shapes"] = shapes
    f["_lag_p"] = lag_p; f["_fw"] = fw; f["_psd"] = psd; f["_bw_mask"] = bw_mask
    return f


# ══════════════════════════════════════════════════════════════════════════════
# STYLE HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _style():
    plt.rcParams.update({
        "font.family":       FONT_FAMILY,
        "font.size":         FONT_SIZE,
        "axes.labelsize":    LABEL_SIZE,
        "axes.titlesize":    FONT_SIZE,
        "xtick.labelsize":   LABEL_SIZE,
        "ytick.labelsize":   LABEL_SIZE,
        "axes.spines.top":   False,
        "axes.spines.right": False,
        "axes.grid":         True,
        "grid.alpha":        0.3,
        "grid.linestyle":    "--",
        "figure.dpi":        DPI_PNG,
    })

def _save(fig: plt.Figure, out_dir: Path, name: str) -> None:
    stem = out_dir / name
    fig.savefig(str(stem) + ".pdf", bbox_inches="tight")
    fig.savefig(str(stem) + ".png", dpi=DPI_PNG, bbox_inches="tight")
    plt.close(fig)
    log.info("  saved %s.{pdf,png}", name)


# ══════════════════════════════════════════════════════════════════════════════
# INDIVIDUAL FIGURE FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def fig_motion_signal(sig, feat, out_dir, stem):
    """s(t) with detected peaks and valleys."""
    seg = sig["diff"]; fps = sig["fps"]
    t = np.arange(len(seg)) / fps
    peaks = feat["_peaks"]; valleys = feat["_valleys"]

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(t, seg, lw=0.8, color=C, alpha=0.9)
    ax.plot(t[peaks],   seg[peaks],   "v", color="k",    ms=3, alpha=0.5, label="Peak")
    ax.plot(t[valleys], seg[valleys], "^", color="gray", ms=2.5, alpha=0.4, label="Valley")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel(r"$s(t)$ (normalised)")
    ax.set_xlim(0, t[-1])
    ax.legend(fontsize=9, framealpha=0.6)
    fig.tight_layout()
    _save(fig, out_dir, f"{stem}_motion_signal")


def fig_motion_zoom(sig, feat, out_dir, stem):
    """~10 cycle zoom of s(t)."""
    seg = sig["diff"]; fps = sig["fps"]; dom = feat["_dom"]
    n_cyc = 10
    zoom = min(int(n_cyc / dom * fps) if dom > 0 else int(fps*3), len(seg))
    tz = np.arange(zoom) / fps
    pkz, _ = find_peaks(seg[:zoom], distance=max(1, int(PEAK_MIN_DIST_SEC*fps)), prominence=seg.std()*PEAK_PROM_FACTOR)
    vlz, _ = find_peaks(-seg[:zoom], distance=max(1, int(PEAK_MIN_DIST_SEC*fps)))

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(tz, seg[:zoom], lw=1.5, color=C)
    ax.fill_between(tz, seg[:zoom], alpha=0.15, color=C)
    ax.plot(tz[pkz], seg[:zoom][pkz], "v", color="k",    ms=6, zorder=5, label="Peak")
    ax.plot(tz[vlz], seg[:zoom][vlz], "^", color="gray", ms=5, zorder=5, label="Valley")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel(r"$s(t)$")
    ax.legend(fontsize=9)
    fig.tight_layout()
    _save(fig, out_dir, f"{stem}_motion_zoom")


def fig_psd(sig, feat, out_dir, stem):
    """Welch PSD."""
    fps = sig["fps"]; seg = sig["diff"]
    fw = feat["_fw"]; psd = feat["_psd"]; bw_mask = feat["_bw_mask"]
    dom = feat["_dom"]; lo = feat["_lo"]; hi = feat["_hi"]

    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.semilogy(fw[bw_mask], psd[bw_mask], color=C, lw=2)
    ax.axvspan(lo, hi, alpha=0.15, color=C, label=f"Band {lo:.1f}–{hi:.1f} Hz")
    ax.axvline(dom, color="k", lw=1.2, ls=":", label=f"$f_{{\\mathrm{{dom}}}}={dom:.2f}$ Hz")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("PSD")
    ax.set_xlim(FREQ_LO_HZ, FREQ_HI_HZ)
    ax.legend(fontsize=9)
    ax.grid(True, which="both", alpha=0.2)
    fig.tight_layout()
    _save(fig, out_dir, f"{stem}_psd")


def fig_freq_stability(sig, feat, out_dir, stem):
    """Per-window dominant frequency over time."""
    tv_f = feat["_tv_f"]; lo = feat["_lo"]; hi = feat["_hi"]
    tv_t = np.arange(len(tv_f)) * FREQ_STEP_SEC
    colours = [C if lo <= v <= hi else C_ANOM for v in tv_f]

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.scatter(tv_t, tv_f, c=colours, s=20, zorder=3)
    ax.plot(tv_t, tv_f, lw=0.6, color="#9E9E9E", zorder=2)
    ax.axhspan(lo, hi, alpha=0.12, color=C, label="Normal band")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("$f_{\\mathrm{dom}}$ (Hz)")
    ax.set_ylim(0, FREQ_HI_HZ + 2)
    ax.legend(fontsize=9)
    fig.tight_layout()
    _save(fig, out_dir, f"{stem}_freq_stability")


def fig_interstroke_interval(sig, feat, out_dir, stem):
    """Histogram of inter-stroke intervals."""
    ivls = feat["_ivls"]
    if len(ivls) < 2:
        return

    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.hist(ivls, bins=40, color=C, alpha=0.8, edgecolor="white")
    ax.axvline(ivls.mean(), color="k", lw=2, label=f"Mean = {ivls.mean():.1f} ms")
    ax.axvline(ivls.mean() - 2*ivls.std(), color=C_ANOM, lw=1, ls="--")
    ax.axvline(ivls.mean() + 2*ivls.std(), color=C_ANOM, lw=1, ls="--")
    ax.set_xlabel("Inter-stroke interval (ms)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)
    fig.tight_layout()
    _save(fig, out_dir, f"{stem}_interstroke_interval")


def fig_stroke_amplitude(sig, feat, out_dir, stem):
    """Histogram of stroke amplitudes."""
    amps = feat["_amps"]

    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.hist(amps, bins=35, color=C, alpha=0.8, edgecolor="white")
    ax.axvline(amps.mean(), color="k", lw=2, label=f"Mean = {amps.mean():.4f}")
    ax.set_xlabel("Stroke amplitude")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)
    fig.tight_layout()
    _save(fig, out_dir, f"{stem}_stroke_amplitude")


def fig_cycle_shape(sig, feat, out_dir, stem):
    """Cycle shape overlay with mean ± std band."""
    shapes = feat["_shapes"]; lag_p = feat["_lag_p"]
    if not shapes:
        log.warning("No cycle shapes available for %s", stem)
        return
    ca = np.array(shapes)
    xc = np.linspace(0, 1, lag_p)

    fig, ax = plt.subplots(figsize=(6, 3.5))
    for row in ca[::max(1, len(ca)//40)]:
        ax.plot(xc, row, color=C, lw=0.5, alpha=0.25)
    ax.plot(xc, ca.mean(axis=0), color="k", lw=2, label="Mean cycle")
    ax.fill_between(xc, ca.mean(axis=0) - ca.std(axis=0),
                        ca.mean(axis=0) + ca.std(axis=0),
                    alpha=0.2, color=C, label=r"$\pm 1\,\sigma$")
    ax.set_xlabel("Normalised position")
    ax.set_ylabel(r"$s(t)$")
    ax.legend(fontsize=9)
    fig.tight_layout()
    _save(fig, out_dir, f"{stem}_cycle_shape")


def fig_phase_portrait(sig, feat, out_dir, stem):
    """Phase portrait s(t) vs s(t+T0)."""
    seg = sig["diff"]; lag_p = feat["_lag_p"]
    if lag_p >= len(seg):
        return

    fig, ax = plt.subplots(figsize=(4.5, 4))
    sc = ax.scatter(seg[:-lag_p], seg[lag_p:],
                    c=np.arange(len(seg)-lag_p), cmap="viridis",
                    s=1.5, alpha=0.3, rasterized=True)
    ax.set_xlabel(r"$s(t)$")
    ax.set_ylabel(r"$s(t + T_0)$")
    plt.colorbar(sc, ax=ax, label="Frame index")
    fig.tight_layout()
    _save(fig, out_dir, f"{stem}_phase_portrait")


def fig_top_bot(sig, feat, out_dir, stem):
    """Top vs bottom ROI motion signals."""
    top = sig["top"]; bot = sig["bot"]; fps = sig["fps"]
    T = len(top); t = np.arange(T) / fps

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(t, top / (top.max() + 1e-9), lw=0.8, color="#E91E63", alpha=0.85, label="Top")
    ax.plot(t, bot / (bot.max() + 1e-9), lw=0.8, color="#00897B", alpha=0.85, label="Bottom")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Normalised motion")
    ax.set_xlim(0, t[-1])
    ax.legend(fontsize=9)
    fig.tight_layout()
    _save(fig, out_dir, f"{stem}_top_bot_motion")


def fig_variance_map(sig, out_dir, stem):
    """Pixel variance heatmap with active region contour."""
    frames = sig["frames"]
    var_map = frames.var(axis=0)

    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(var_map, cmap="hot", aspect="equal")
    ax.contour((var_map > var_map.mean() * 1.5).astype(float),
               levels=[0.5], colors="cyan", linewidths=1.5)
    plt.colorbar(im, ax=ax, label="Temporal variance")
    ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout()
    _save(fig, out_dir, f"{stem}_variance_map")


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def main():
    _style()

    video_path = Path(VIDEO_PATH)
    out_dir    = Path(OUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = video_path.stem

    # load ROI
    roi_db = json.load(open(ROI_JSON))
    key = stem + ".jpg"
    if key not in roi_db:
        log.error("No ROI entry for key '%s' in %s", key, ROI_JSON)
        return
    x, y, w, h = roi_db[key]
    roi = (int(x), int(y), int(x+w), int(y+h))

    log.info("Loading signals from %s ...", video_path.name)
    sig = load_signals(str(video_path), roi)
    if sig is None:
        log.error("Signal extraction failed.")
        return

    log.info("Computing features ...")
    feat = compute_features(sig)
    log.info("dom=%.2f Hz  band_pwr=%.3f  interval_cv=%.3f",
             feat["dominant_freq_hz"], feat["spectral_power_in_band"], feat["interval_cv"])

    log.info("Saving figures to %s ...", out_dir)
    fig_motion_signal(sig, feat, out_dir, stem)
    fig_motion_zoom(sig, feat, out_dir, stem)
    fig_psd(sig, feat, out_dir, stem)
    fig_freq_stability(sig, feat, out_dir, stem)
    fig_interstroke_interval(sig, feat, out_dir, stem)
    fig_stroke_amplitude(sig, feat, out_dir, stem)
    fig_cycle_shape(sig, feat, out_dir, stem)
    fig_phase_portrait(sig, feat, out_dir, stem)
    fig_top_bot(sig, feat, out_dir, stem)
    fig_variance_map(sig, out_dir, stem)

    log.info("Done. %d figures saved.", 10)


if __name__ == "__main__":
    main()